In [202]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

df = pd.read_csv('C:\\FUFA Code\\Match-Analysis\\data\\raw\\UPL25_26_midseason_raw_catapult_data.csv')

In [203]:
df.shape

(20866, 100)

In [204]:
# strip whitespace, replace spaces/slashes with underscores
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(r'\s+', '_', regex=True)
      .str.replace(r'[\/\(\)\-]', '', regex=True)
)

In [205]:

def clean_text(s):
    if pd.isnull(s):
        return s
    s = s.strip()                        # Remove leading/trailing whitespace
    s = re.sub(r'\s+', ' ', s)          # Replace multiple spaces with single
    s = s.lower()                       # Convert to lowercase (optional)
    # s = s.replace('_', ' ')             # Replace underscores with space
    s = s.title()                       # Convert to Title Case

    return s

# Apply cleaning to each text column
cat_cols = [col for col in df.columns if df[col].dtype == 'object']
for col in cat_cols:
    df[col] = df[col].astype(str).apply(clean_text)

In [206]:
#convert date to datetime format
df['date'] = pd.to_datetime(df['date'], dayfirst=True, errors='coerce')

In [207]:
#select only the 'Game' tag, exclude training etc 
df = df[((df['tags'] == 'Game') | (df['tags'] == 'Game Training'))] #game training is a system glitch but also covers game data 
df = df.drop('tags', axis=1)

In [208]:
df['duration'] = df['duration'] / 60 #convert duration to minutes for easy comparison

In [209]:
# #select only the first and second halves ; ensure proper naming of splits
# df = df[((df['split_name'] == '1St.Half') | (df['split_name'] == '2Nd.Half'))]
# df['split_name'] = df['split_name'].str.replace('1St.Half', '1st Half')
# df['split_name'] = df['split_name'].str.replace('2Nd.Half', '2nd Half')

In [210]:
#select only the first and second halves ; ensure proper naming of splits
df = df[((df['split_name'] == '1St.Half') | (df['split_name'] == '2Nd.Half') | (df['split_name'] == 'All'))]
df['split_name'] = df['split_name'].str.replace('1St.Half', '1st Half')
df['split_name'] = df['split_name'].str.replace('2Nd.Half', '2nd Half')

In [211]:
df['split_name'].unique()

array(['1st Half', '2nd Half', 'All'], dtype=object)

In [212]:
# Select only session_title entries matching the pattern: 'Md1-Kcca Fc-Ura Fc-Home-League-Win' ( standard naming convention)
pattern = r'^Md\d+-[\w\s\.]+-[\w\s\.]+-[\w\s\.]+-[\w\s\.]+-[\w\s\.]+$'
df = df[df['session_title'].str.match(pattern, na=False)]

In [213]:
df.shape

(9538, 99)

In [214]:

# 1. Clean the session title
df['session_title'] = (
    df['session_title']
    .str.strip()
    .str.replace(r'\.+', '', regex=True)         # remove dots like "F.C."
    .str.replace(r'\s*-\s*', '-', regex=True)    # normalize spacing around hyphens
    .str.replace(r'\s+', ' ', regex=True)        # reduce internal whitespace
    .str.title()                                 # ensure proper capitalization
)

# 2. Extract session components: 6 fields + ignore trailing data
regex = re.compile(
    r'^(Md\d+)-'            # Matchday
    r'(.+?)-'               # Club For
    r'(.+?)-'               # Club Against
    r'(.+?)'                # Part 1: location/league/result
    r'[-\s]+(.+?)'          # Part 2: location/league/result
    r'[-\s]+(.+?)'          # Part 3: location/league/result
    r'(?:\s|$)',            # Ignore trailing info
    re.IGNORECASE
)
extracted = df['session_title'].str.extract(regex)

# 3. Assign temporary column names
extracted.columns = [
    'match_day',
    'club_for',
    'club_against',
    'part1',
    'part2',
    'part3'
]

# 4. Define sets to identify
location_set = {'Home', 'Away'}
league_set   = {'League', 'Cup'}
result_set   = {'Win', 'Loss', 'Draw'}

# 5. Assign actual values to correct columns
extracted['location'] = None
extracted['league']   = None
extracted['result']   = None

for i, row in extracted.iterrows():
    for val in [row['part1'], row['part2'], row['part3']]:
        if pd.isna(val): continue
        val_clean = val.strip().title()
        if val_clean in location_set:
            extracted.at[i, 'location'] = val_clean
        elif val_clean in league_set:
            extracted.at[i, 'league'] = val_clean
        elif val_clean in result_set:
            extracted.at[i, 'result'] = val_clean

# 6. Keep only fully matched and valid rows
valid = extracted.dropna(subset=['location', 'league', 'result']).copy()

# 7. Finalize merge
df = df.loc[valid.index].reset_index(drop=True)
valid = valid.drop(columns=['part1', 'part2', 'part3']).reset_index(drop=True)
df = pd.concat([df.reset_index(drop=True), valid], axis=1)


In [215]:
df.shape

(9370, 105)

In [216]:
# Extract player name, club, and position from 'player_name' column

# STEP 1: Fix incorrect entries where second separator is an underscore
# This converts 'First Last - Club_Position' → 'First Last - Club - Position'
def fix_player_name_format(name):
    if re.fullmatch(r'.+ - .+_.+', name):
        return re.sub(r'(.+ - .+?)_(.+)', r'\1 - \2', name)
    return name

df['player_name'] = df['player_name'].apply(fix_player_name_format)

# STEP 2: Extract the fields from the fixed format
player_regex = re.compile(
    r'^(.+?)\s*-\s*'     # Player name
    r'(.+?)\s*-\s*'      # Club
    r'(.+?)$'            # Position
)

player_cols = df['player_name'].str.extract(player_regex)
player_cols.columns = ['p_name', 'player_club_', 'player_position']

# STEP 3: Keep only rows where all 3 groups were found
valid = player_cols.dropna().copy()
df = df.loc[valid.index].reset_index(drop=True)
player_cols = player_cols.loc[valid.index].reset_index(drop=True)

# STEP 4: Add extracted columns back
df = pd.concat([df, player_cols], axis=1)


In [217]:
df.shape

(9370, 108)

In [218]:
# #split the player name and session title columns into columns 
# df[['match_day', 'club_for', 'club_against', 'location', 'league', 'result']] = df['session_title'].str.split('-', n=5, expand=True)
# df[['p_name', 'player_club_', 'player_position']] = df['player_name'].str.split('-', n=2, expand=True)
df = df.drop(['session_title', 'player_name'], axis=1)

# Apply cleaning to each text column again to catch any new text data
cat_cols = [col for col in df.columns if df[col].dtype == 'object']
for col in cat_cols:
    df[col] = df[col].astype(str).apply(clean_text)

In [219]:
#ensure all entries have the proper 'league' value; fix any that may be interchanged with location or result 
mask = df['league'] != 'League'
df.loc[mask, ['location', 'league']] = df.loc[mask, ['league', 'location']].values
df.loc[df['league'] == 'Home', 'league'] = 'League'
df = df.drop('league', axis=1)

In [220]:
#ensure proper location values; switch any interchanged and fill any missed one 
df.loc[df['location'] == 'Draw', ['location', 'result']] = ['Home', 'Draw']
df.loc[df['location'] == 'Upl', 'location'] = np.nan
df.loc[df['location'] == 'Way', ['location', 'result']] = ['Away', np.nan]
# df.loc[df['location'].isna(), 'location'] = 'Home' # it was only maroons (md16) without a location - filled manually after google search 

In [221]:
# df.loc[df['result'].isna(), 'result'] = 'Win' # it was only soltilo (md29) without a result - filled manually after a google search

In [222]:
df.shape

(9370, 105)

In [223]:
# list of standard UPL club names
standard_clubs = [
    "KCCA FC", "BUL FC", "URA FC", "Vipers SC", "Express FC", "SC Villa",
    "Maroons FC", "Buhimba United Saints FC", "Entebbe UPPC FC", "Police FC",
    "UPDF FC", "NEC FC", "Mbarara City FC", "Kitara FC",'Lugazi FC','Calvary FC'
]
def normalize_name(name):
    # Remove spaces, punctuation, and 'fc', 'sc', etc.
    name = name.lower()
    name = re.sub(r'[^a-z]', '', name)  # keep only letters
    name = name.replace('fc', '').replace('sc', '')
    return name

def best_match(name, club_list, min_score=0.6):
    name_clean = name.strip().lower().replace('.', '')
    norm_name = normalize_name(name_clean)
    # 1. Normalized exact match
    for club in club_list:
        if norm_name == normalize_name(club):
            return club
    # 2. Normalized substring match
    for club in club_list:
        club_norm = normalize_name(club)
        if norm_name in club_norm or club_norm in norm_name:
            return club
    # 3. Token overlap (as before)
    name_tokens = set(name_clean.split())
    best = None
    best_score = 0
    for club in club_list:
        club_tokens = set(club.strip().lower().replace('.', '').split())
        score = len(name_tokens & club_tokens) / max(len(club_tokens), 1)
        if score > best_score:
            best = club
            best_score = score
    return best if best_score >= min_score else name

# Apply to all columns with club names 
for col in ['club_for', 'club_against', 'player_club_']:
    if col in df.columns:
        df[col] = df[col].astype(str).apply(lambda x: best_match(x, standard_clubs))


In [224]:
df.shape

(9370, 105)

In [225]:
for col in ['club_for', 'club_against', 'player_club_']:
    if col in df.columns:
        # df[col] = df[col].replace('Solitilo Bright Stars', 'Soltilo Bright Stars FC')
        # df[col] = df[col].replace('Soltito Bright Stars', 'Soltilo Bright Stars FC')
        # df[col] = df[col].replace('Solitilo', 'Soltilo Bright Stars FC')
        # df[col] = df[col].replace('Soliyilo', 'Soltilo Bright Stars FC')
        df[col] = df[col].replace('Lugzi', 'Lugazi FC')

In [226]:
# Drop rows where entries of  any of the specified columns are not in standard_clubs -remove female clubs
cols_to_check = ['club_for', 'club_against', 'player_club_']
mask = df[cols_to_check].apply(lambda x: all(val in standard_clubs for val in x), axis=1)
df = df[mask]

In [227]:
df.shape

(9352, 105)

In [228]:
# df[df['club_for'] != df['player_club_']][['p_name','club_for', 'player_club_']]
#leave both the 'club_for' and 'player_club_' columnns , could be useful for transfer analysis

In [229]:
missing_positions={
    'Saidi Mayanja':'CM',
    'Ashraf Mugume':'CM',
    'Musitafa Mujuzi':'CB',
    'Bright Anukani':'AM',
    'Kiza Arafat':'AM',
    'Joel Sserunjogi':'DM',
    'Katenga Etienne Openga':'LW',
    'Hassan Muhamud':'CB',
    'Derrick Paul':'LW',
    'Emmanuel Anyama':'CF',
    'Abubaker Mayanja':'CF',
    'Isa Mubiru':'LB',
    'Julius Poloto':'MD',
    'Peter Magambo':'DM',
}

In [230]:
# Replace None player_position using missing_positions and p_name
mask = df['player_position'] == 'None'
df.loc[mask, 'player_position'] = df.loc[mask, 'p_name'].map(missing_positions).fillna('None')

In [231]:
df['player_position'] = df['player_position'].str.lower()

In [232]:
df['general_position'] = df['player_position'].map({
            'gk': 'goalkeeper', 'df': 'defender', 'mf': 'midfielder', 'am': 'midfielder',
            'fw': 'forward', 'rb': 'defender', 'cb': 'defender', 'lb': 'defender',
            'rw': 'forward', 'lw': 'forward', 'cm': 'midfielder', 'dm': 'midfielder',
            'cd':'defender','fwd':'forward','dmc':'midfielder','amc':'midfielder','dc':'defender',
            'cf':'forward','mc':'midfielder','lcb':'defender','md':'midfielder'
        })

In [233]:
df['general_position'].unique()

array(['defender', 'midfielder', 'forward', 'goalkeeper'], dtype=object)

In [234]:
df = df[df['player_position'] != 'gk']

In [235]:
df.shape

(8866, 106)

In [236]:
# Apply cleaning to each text column again to catch any new text data
cat_cols = [col for col in df.columns if df[col].dtype == 'object']
for col in cat_cols:
    df[col] = df[col].astype(str).apply(clean_text)
    df[col] = df[col].astype('category')

In [237]:
zero_frac = (df == 0).mean().sort_values(ascending=False)
sparse = zero_frac[zero_frac > 0.95]
sparse.index.tolist()

# #drop these columns(more than 95% of their values are 0) from the dataset
df = df.drop(columns=sparse.index.tolist())

In [238]:
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns#all numeric columns
num_cols = ['duration','distance_km','sprint_distance_m','power_plays','energy_kcal','impacts',
                   'player_load','top_speed_kmh','distance_per_min_mmin','power_score_wkg','work_ratio','max_acceleration_mss','max_deceleration_mss']# numeric columns of high interest

In [239]:
#Identify the columns with missing data
missing_counts = df.isnull().sum().sort_values(ascending=False)
print(f'Number of columns with missing data: {len(missing_counts[missing_counts > 0])}')
# Check for duplicate rows in the dataset
duplicate_rows = df.duplicated()
print(f"Number of duplicate rows: {duplicate_rows.sum()}")
df = df[~duplicate_rows]  # Remove duplicate rows
print('Duplicates removed')

df = df.drop(columns=['split_start_time', 'split_end_time'])


Number of columns with missing data: 0
Number of duplicate rows: 33
Duplicates removed


In [240]:
df.head(5)

,date,split_name,duration,distance_km,sprint_distance_m,power_plays,energy_kcal,impacts,player_load,top_speed_kmh,...,deceleration_zone_count:_>_4_mss,match_day,club_for,club_against,location,result,p_name,player_club_,player_position,general_position
0,1970-01-01 00:00:00.000046004,All,99.166667,8.8100,680.143,58,1082.0286,17,445.3572,31.0322,...,53,Md10,Bul Fc,Vipers Sc,Away,Loss,Hillary Onek,Bul Fc,Rb,Defender
1,1970-01-01 00:00:00.000046004,All,100.183333,8.7486,823.990,66,1402.7717,12,413.5575,32.2723,...,43,Md10,Bul Fc,Vipers Sc,Away,Loss,Nicholas Mwere,Bul Fc,Lb,Defender
2,1970-01-01 00:00:00.000046004,All,99.266667,7.6467,469.084,35,1045.4796,10,430.3503,31.1442,...,28,Md10,Bul Fc,Vipers Sc,Away,Loss,Walter Ochora,Bul Fc,Cb,Defender
3,1970-01-01 00:00:00.000046004,All,77.616667,8.0449,313.038,49,1007.8479,10,384.1151,26.0802,...,37,Md10,Bul Fc,Vipers Sc,Away,Loss,George Kasonko,Bul Fc,Dm,Midfielder
4,1970-01-01 00:00:00.000046004,All,23.283333,2.4059,73.712,15,292.3570,5,116.6583,22.9842,...,4,Md10,Bul Fc,Vipers Sc,Away,Loss,Ibrahim Mugulusi,Bul Fc,Am,Midfielder


In [241]:
df.shape

(8833, 91)

In [242]:
raw_counts = df.groupby(['club_for', 'match_day'],observed=False)['p_name'].nunique().reset_index(name='raw_unique_players')
raw_counts[raw_counts['club_for'] == 'Kcca Fc'].sort_values('match_day')

,club_for,match_day,raw_unique_players
60,Kcca Fc,Md1,17
61,Kcca Fc,Md10,13
62,Kcca Fc,Md11,14
63,Kcca Fc,Md12,14
64,Kcca Fc,Md13,15
65,Kcca Fc,Md14,15
66,Kcca Fc,Md15,16
67,Kcca Fc,Md2,15
68,Kcca Fc,Md3,14
69,Kcca Fc,Md4,14


In [243]:
def describe_numeric_columns(df):
    n_desc = df[num_cols].describe().T
    n_desc = n_desc.drop(['25%', '75%'], axis=1)
    n_desc = n_desc.rename(columns={'50%': 'median'})
    display(n_desc.T[num_cols].T)

def describe_categorical_columns(df):
    cat_vars = df.select_dtypes(include='category').columns
    c_desc = df[cat_vars].describe().T
    c_desc['%age'] = c_desc['freq'] / len(df) * 100
    display(c_desc)
    
def plot_numerical_distributions(df):
    plt.figure(figsize=(16, 8))
    for i, col in enumerate(num_cols):
        plt.subplot(4, 4, i + 1)
        sns.histplot(df[col], bins=30, kde=True)
        plt.title(f'Distribution of {col}')
        plt.xlabel(col)
        plt.ylabel('Frequency')
    plt.tight_layout()



In [244]:
half2_df = df[df['split_name'] == '2Nd Half']
half2_df = half2_df.drop('split_name', axis=1)

half1_df = df[df['split_name'] == '1St Half']
half1_df = half1_df.drop('split_name', axis=1)

all_df = df[df['split_name'] == 'All']
all_df = all_df.drop('split_name', axis=1)

In [245]:
len(half1_df), len(half2_df), len(all_df)

(2931, 2930, 2972)

In [ ]:
merge_keys = ['p_name', 'club_for', 'club_against', 'player_club_','match_day','general_position','player_position','result','location']


half1_df_rn = half1_df.rename(columns=lambda x: x + '_H1' if x not in merge_keys else x)
half2_df_rn = half2_df.rename(columns=lambda x: x + '_H2' if x not in merge_keys else x)
all_df_rn = all_df.rename(columns=lambda x: x + '_ALL' if x not in merge_keys else x)

df_merged = pd.merge(half1_df_rn, half2_df_rn, on=merge_keys, how='outer')
df_merged = pd.merge(df_merged, all_df_rn, on=merge_keys, how='outer')


df_combined = df_merged[merge_keys].copy()

for col in num_cols:
    # Skip metrics that need special handling
    if col not in ['top_speed_kmh', 'distance_per_min_mmin', 'max_acceleration_mss', 
                   'max_deceleration_mss', 'work_ratio', 'power_score_wkg']:
        h1 = f"{col}_H1"
        h2 = f"{col}_H2"
        df_combined[col] = df_merged[h1].fillna(0) + df_merged[h2].fillna(0)

# Top Speed - consider max
df_combined['top_speed_kmh'] = df_merged[['top_speed_kmh_H1', 'top_speed_kmh_H2']].max(axis=1)
# Max Acceleration - consider max
df_combined['max_acceleration_mss'] = df_merged[['max_acceleration_mss_H1', 'max_acceleration_mss_H2']].max(axis=1)
# Max Deceleration - consider max
df_combined['max_deceleration_mss'] = df_merged[['max_deceleration_mss_H1', 'max_deceleration_mss_H2']].max(axis=1)
# distance per minute - recalculate from sum
df_combined['distance_per_min_mmin'] = (df_combined['distance_km'] *1000) / df_combined['duration']
#work ratio - take the one from all if it exists, otherwise put nan - cant be calcualted from halves by defintion
df_combined['work_ratio'] = df_merged['work_ratio_ALL']
#power score - take the one from all if it exists, otherwise calculate from halves-also cant be calculated from halves by definiion
df_combined['power_score_wkg'] = df_merged['power_score_wkg_ALL']

#all other numerical variables - consider sum
# Columns in numeric_cols that are not in num_cols
for col in numeric_cols.difference(num_cols):
    if col not in merge_keys:
        h1 = f"{col}_H1"
        h2 = f"{col}_H2"
        # Only sum if both columns exist in df_merged
        if h1 in df_merged.columns and h2 in df_merged.columns:
            df_combined[col] = df_merged[h1].fillna(0) + df_merged[h2].fillna(0)


In [249]:
match_df = df_combined

In [250]:
# Apply cleaning to each text column
cat_cols = [col for col in match_df.columns if match_df[col].dtype == 'object']
for col in cat_cols:
    match_df[col] = match_df[col].astype(str).apply(clean_text)

In [251]:
dur_counts = match_df.groupby(['club_for', 'match_day'],observed=False)['duration'].mean().reset_index(name='avg_duration')
dur_counts[dur_counts['club_for'] == 'Sc Villa'].sort_values('match_day')

raw_counts = match_df.groupby(['club_for', 'match_day'],observed=False)['p_name'].nunique().reset_index(name='raw_unique_players')
raw_counts[raw_counts['club_for'] == 'Police Fc'].sort_values('match_day')

,club_for,match_day,raw_unique_players
150,Police Fc,Md1,14
151,Police Fc,Md10,10
152,Police Fc,Md11,11
153,Police Fc,Md12,10
154,Police Fc,Md13,12
155,Police Fc,Md14,13
156,Police Fc,Md15,10
157,Police Fc,Md2,12
158,Police Fc,Md3,12
159,Police Fc,Md4,12


In [252]:
dur_counts = match_df.groupby(['club_for', 'match_day'],observed=False)[['duration','distance_km']].mean().reset_index()
dur_counts[dur_counts['club_for'] == 'Police Fc'].sort_values('match_day')

,club_for,match_day,duration,distance_km
150,Police Fc,Md1,73.129762,6.386357
151,Police Fc,Md10,99.561667,7.592980
152,Police Fc,Md11,99.593939,8.937764
153,Police Fc,Md12,102.286667,10.221550
154,Police Fc,Md13,80.416667,7.035633
155,Police Fc,Md14,89.041026,6.144300
156,Police Fc,Md15,86.428333,7.600740
157,Police Fc,Md2,75.418056,6.897225
158,Police Fc,Md3,79.038889,6.897642
159,Police Fc,Md4,79.961111,7.194473


In [253]:
# Thresholds
DIST_THRESH = 2   # players should cover at least 2km in a session to count
DURATION_THRESH = 60  # players should be active for at least 60 minutes in a session    


# Extend your active_session flag:
match_df['active_session'] = (
    (match_df['distance_km']    >= DIST_THRESH)  
    & (match_df['duration']  >= DURATION_THRESH)   
)

match_df = match_df[match_df['active_session'] == True]  # Filter main df for active sessions


In [254]:
match_df = match_df.drop('active_session', axis=1)  # Drop the 'active_session' column after filtering

In [255]:
# Distance IQR
q1_dist = match_df['distance_km'].quantile(0.25)
q3_dist = match_df['distance_km'].quantile(0.75)
iqr_dist = q3_dist - q1_dist
lower_bound_dist = q1_dist - 1.5 * iqr_dist
upper_bound_dist = q3_dist + 1.5 * iqr_dist

# Duration IQR
q1_dur = match_df['duration'].quantile(0.25)
q3_dur = match_df['duration'].quantile(0.75)
iqr_dur = q3_dur - q1_dur
lower_bound_dur = q1_dur - 1.5 * iqr_dur
upper_bound_dur = q3_dur + 1.5 * iqr_dur

# Filter out outliers
match_df = match_df[
    (match_df['distance_km'] >= lower_bound_dist) & (match_df['distance_km'] <= upper_bound_dist) &
    (match_df['duration']    >= lower_bound_dur)  & (match_df['duration']    <= upper_bound_dur)
]


In [256]:
print(f"Duration bounds: {lower_bound_dur:.2f} to {upper_bound_dur:.2f}")
print(f"Distance bounds: {lower_bound_dist:.2f} to {upper_bound_dist:.2f}")


Duration bounds: 82.32 to 111.33
Distance bounds: 2.81 to 14.68


In [257]:
match_df['duration'].describe()

count    1872.000000
mean       97.746368
std         4.915242
min        82.450000
25%        95.016667
50%        98.050000
75%       100.858333
max       110.916667
Name: duration, dtype: float64

In [260]:
describe_numeric_columns(match_df)

,count,mean,std,min,median,max
duration,1872.0,97.746368,4.915242,82.450000,98.050000,110.916667
distance_km,1872.0,8.796929,2.293265,2.822300,9.443100,13.935500
sprint_distance_m,1872.0,850.172096,368.845443,7.531000,813.790500,2282.068000
power_plays,1872.0,63.556090,22.664284,4.000000,63.000000,136.000000
energy_kcal,1872.0,1083.149298,291.725243,301.116500,1157.034500,1942.324500
impacts,1872.0,6.697115,5.156620,0.000000,6.000000,41.000000
player_load,1872.0,403.115531,96.828219,137.994300,427.251400,639.953600
top_speed_kmh,1872.0,29.925548,2.142486,19.632200,29.968200,35.704300
distance_per_min_mmin,1872.0,90.108632,23.448256,27.145551,96.752734,141.238176
power_score_wkg,1872.0,4.507757,1.584083,0.869200,4.589000,8.765700


In [261]:
raw_counts = match_df.groupby(['club_for', 'match_day'],observed=False)['p_name'].nunique().reset_index(name='raw_unique_players')
raw_counts[raw_counts['club_for'] == 'Updf Fc'].sort_values('match_day')


,club_for,match_day,raw_unique_players
180,Updf Fc,Md1,10
181,Updf Fc,Md10,9
182,Updf Fc,Md11,11
183,Updf Fc,Md12,10
184,Updf Fc,Md13,10
185,Updf Fc,Md14,10
186,Updf Fc,Md15,10
187,Updf Fc,Md2,10
188,Updf Fc,Md3,11
189,Updf Fc,Md4,10


In [ ]:
# Save the cleaned and processed DataFrame to a new CSV file
# match_df.to_csv('C:\\FUFA Code\\Match-Analysis\\data\\processed\\UPL_25_26_mid_season_matches_cleaned.csv',index=False)